# **Dependencies**

In [ ]:
# Starting library
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# nltk dependencies
nltk.download('stopwords')
nltk.download('punkt')

In [ ]:
# Dataset
slang_url = 'https://raw.githubusercontent.com/okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection/master/new_kamusalay.csv'
dataset_url = 'https://raw.githubusercontent.com/okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection/master/re_dataset.csv'

df_slang = pd.read_csv(slang_url, encoding = 'latin1', header = None)
df_slang.columns = ['original', 'swap']
df = pd.read_csv(dataset_url, encoding = 'latin1')

In [ ]:
# Helper function 
def check(word):
  return df[df['Tweet'].str.contains(word, case = False)]['Tweet'].sample(10).unique()

# Words removal function
def remove_words(sentence, word_list):
  word_tokens = word_tokenize(sentence)
    
  filtered = [w for w in word_tokens if w.lower() not in word_list]
    
  new = ' '.join(filtered)
    
  return new

# Words replace function
def replace_words(sentence, dictionary):
  word_list = dictionary.keys()
  word_tokens = word_tokenize(sentence)

  replaced = [dictionary[w] if w.lower() in list(dictionary.keys()) else w for w in word_tokens]

  new = ' '.join(replaced)

  return new

In [ ]:
# Slang words 
df_slang.head()

In [ ]:
# Dataset
df.head()

# **Cleansing**

In [ ]:
# Explore, sampling
df['Tweet'].sample(20).unique()

### Regex cleaning

In [ ]:
# Cleaning text using regex
def clean_text(Tweet):
    Tweet = re.sub('rt\s', '', Tweet)   # remove "rt" -> "retweet"
    Tweet = re.sub('x[\w]+', '', Tweet)    # remove xf, xa etc.
    Tweet = re.sub('[^a-zA-Z]+', ' ', Tweet)  # remove non-alphabet
    Tweet = re.sub('((www\.[^\s]+)|(https?://[^\s]+)|(http?://[^\s]+))', '', Tweet)   #remove link elements
    Tweet = re.sub('wk+', '', Tweet)    # remove "wkwk" -> indonesian laughing words
    Tweet = re.sub('[akwjhz]{3,}', '', Tweet)     # remove multiple occuring single alphabet noise "aaaa" "hhhh" etc.
    return Tweet

### Slang words

In [ ]:
# Create dictionary from slang words df
slang_dict = pd.Series(df_slang['swap'].values, index = df_slang['original']).to_dict()
for i in list(slang_dict.keys())[:10]:
  print(i, ':', slang_dict[i])

In [ ]:
# Test replace words function
test_sen = "abis ini kita bkal pegi ke pasar"
replace_words(test_sen, slang_dict)

### Stop words

In [ ]:
# Stop words
stop_words = stopwords.words('indonesian')

# Stop words sample
stop_words[:10]

In [ ]:
# Add more words to remove

other_words = ['si', 'an', 'amp','user', 'pengguna', 'url', 'ru', 'uniform', 'resource', 'locator' ,'ya', 'iya', 'sih', 'nya', 'nih', 'banget', 'kayak']

character = [chr(i) for i in range(ord('a'), ord('z') + 1)]

stop_words_join = stop_words + other_words + character

In [ ]:
# Test remove stop word function
test_sen = "saya mau pergi ke sekolah naik mobil balap"
new_sen = remove_words(test_sen, stop_words)
new_sen

### Cleaning pipeline

In [ ]:
# Tweet cleaning pipeline 
df['Tweet'] = df['Tweet'].str.lower()
df['Tweet'] = df['Tweet'].apply(lambda x : clean_text(x))
df['Tweet'] = df['Tweet'].apply(lambda x : replace_words(x, slang_dict))
df['Tweet'] = df['Tweet'].apply(lambda x : remove_words(x, stop_words_join))
df['Tweet'] = df['Tweet'].str.strip()

In [ ]:
# Explore
df['Tweet'].sample(20).unique()

In [ ]:
check('jokowi')

# **Exploration**

### Random Sample

In [ ]:
# Sample HS
df[df['HS'] == 1]['Tweet'].sample(10).unique()

In [ ]:
# Sample non HS
df[df['HS'] == 0]['Tweet'].sample(10).unique()

### Unigram

In [ ]:
# Overall Unigram
unigram = df['Tweet'].str.split(expand = True).stack()
unigram.value_counts().head(30)

In [ ]:
# Hatespeech unigram
hs_con = df['HS'] == 1
hs_unigram = df[hs_con]['Tweet'].str.split(expand = True).stack()
hs_unigram.value_counts().head(30)

In [ ]:
# Non-hatespeech unigram
non_hs_con = df['HS'] == 0
non_hs_unigram = df[non_hs_con]['Tweet'].str.split(expand = True).stack()
non_hs_unigram.value_counts().head(30)

# **Data Selection**

In [ ]:
data = df[['Tweet', 'HS']]
data.columns = ['tweet','hs']
data.head()

In [ ]:
data['hs'].value_counts()

In [ ]:
data['tweet'] = data['tweet'].astype(str)

In [ ]:
data = data.dropna()
data.info()

In [ ]:
# Export to CSV
data.to_csv('/content/drive/MyDrive/Colab Notebooks/hatespeech_dataset.csv', index = False)